In [2]:
%reload_ext autoreload
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import random

import torch, os
from TTS.api import TTS
from TTS.utils.manage import ModelManager

OUT_PATH = "./"
CHECKPOINTS_OUT_PATH = os.path.join(OUT_PATH, "XTTS_v2.0_original_model_files/")
os.makedirs(CHECKPOINTS_OUT_PATH, exist_ok=True)

DVAE_CHECKPOINT_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/dvae.pth"
MEL_NORM_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/mel_stats.pth"

DVAE_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(DVAE_CHECKPOINT_LINK))
MEL_NORM_FILE = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(MEL_NORM_LINK))

# download DVAE files if needed
if not os.path.isfile(DVAE_CHECKPOINT) or not os.path.isfile(MEL_NORM_FILE):
    print(" > Downloading DVAE files!")
    ModelManager._download_model_files([MEL_NORM_LINK, DVAE_CHECKPOINT_LINK], CHECKPOINTS_OUT_PATH, progress_bar=True)

TOKENIZER_FILE_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/vocab.json"
XTTS_CHECKPOINT_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/model.pth"
XTTS_CONFIG_LINK = "https://coqui.gateway.scarf.sh/hf-coqui/XTTS-v2/main/config.json"

TOKENIZER_FILE = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(TOKENIZER_FILE_LINK))  # vocab.json file
XTTS_CHECKPOINT = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(XTTS_CHECKPOINT_LINK))  # model.pth file
XTTS_CONFIG_LINK = os.path.join(CHECKPOINTS_OUT_PATH, os.path.basename(XTTS_CONFIG_LINK))  # model.pth file

# download XTTS v2.0 files if needed
if not os.path.isfile(XTTS_CHECKPOINT) or not os.path.isfile(TOKENIZER_FILE):
    print(" > Downloading XTTS v2.0 files!")
    ModelManager._download_model_files(
        [TOKENIZER_FILE_LINK, XTTS_CHECKPOINT_LINK], CHECKPOINTS_OUT_PATH, progress_bar=True
    )


/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [4]:
# Get device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Init TTS
tts = TTS(
    model_path=CHECKPOINTS_OUT_PATH,
    config_path=os.path.join(CHECKPOINTS_OUT_PATH, "config.json"),
    progress_bar=True
).to(device)

tts.synthesizer.output_sample_rate = 24000

/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/romolo/VT1/coqui-tts/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


 > Using model: xtts


In [5]:
model = tts.synthesizer.tts_model

In [80]:
from TTS.tts.layers.xtts.trainer.gpt_trainer import GPTArgs, GPTTrainer, GPTTrainerConfig, XttsAudioConfig

model_args = GPTArgs(
    max_conditioning_length=132300,  # 6 secs
    min_conditioning_length=66150,  # 3 secs
    debug_loading_failures=False,
    max_wav_length=255995,  # ~11.6 seconds
    max_text_length=200,
    mel_norm_file=MEL_NORM_FILE,
    dvae_checkpoint=DVAE_CHECKPOINT,
    xtts_checkpoint=XTTS_CHECKPOINT,  # checkpoint path of the model that you want to fine-tune
    tokenizer_file=TOKENIZER_FILE,
    gpt_num_audio_tokens=1026,
    gpt_start_audio_token=1024,
    gpt_stop_audio_token=1025,
    gpt_use_masking_gt_prompt_approach=True,
    gpt_use_perceiver_resampler=True,
)
# define audio config
audio_config = XttsAudioConfig(sample_rate=22050, dvae_sample_rate=22050, output_sample_rate=24000)
# training parameters config
config = GPTTrainerConfig(
    output_path=OUT_PATH,
    model_args=model_args,
    epochs=1000,
    run_description="""
        GPT XTTS training
        """,
    audio=audio_config,
    model_param_stats=False,
    batch_size=32,
    batch_group_size=48,
    eval_batch_size=32,
    num_loader_workers=2,
    eval_split_max_size=256,
    eval_split_size=0.02,
    print_step=50,
    plot_step=100,
    log_model_step=1000,
    save_step=1000,
    save_n_checkpoints=1,
    save_checkpoints=True,
    wandb_entity='RM',
    # target_loss="loss",
    print_eval=False,
    datasets=None,
    shuffle=True,
    # Optimizer values like tortoise, pytorch implementation with modifications to not apply WD to non-weight parameters.
    optimizer="AdamW",
    optimizer_wd_only_on_weights=True,
    optimizer_params={"betas": [0.9, 0.96], "eps": 1e-8, "weight_decay": 1e-2},
    lr=6e-05,  # learning rate
    lr_scheduler="MultiStepLR",
    # it was adjusted accordly for the new step scheme
    lr_scheduler_params={"milestones": [50000 * 18, 150000 * 18, 300000 * 18], "gamma": 0.5, "last_epoch": -1},
    use_h5=True,
)

# init the model from config
train_model = GPTTrainer.init_from_config(config)

>> DVAE weights restored from: ./XTTS_v2.0_original_model_files/dvae.pth


In [81]:
import whisper

In [82]:
asr = whisper.load_model('base')

In [173]:
import torchaudio

def crop_audio(input_path, output_path, duration_sec=5):
    waveform, sample_rate = torchaudio.load(input_path)
    num_samples = int(duration_sec * sample_rate)
    cropped_waveform = waveform[:, :num_samples]
    torchaudio.save(output_path, cropped_waveform, sample_rate)

# Example usage:
crop_audio("/home/romolo/VT1/coqui-tts/data/target_no_sr.wav", "/home/romolo/VT1/coqui-tts/data/TARGET_5s.wav", duration_sec=5)
crop_audio("/home/romolo/VT1/coqui-tts/data/REF.wav", "/home/romolo/VT1/coqui-tts/data/REF_5s.wav", duration_sec=5)

In [270]:
import torch
import torchaudio
import noisereduce as nr
import numpy as np
import librosa
import soundfile as sf

def denoise_audio(input_path, output_path):
    # Load audio
    waveform, sample_rate = torchaudio.load(input_path)

    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    audio_np = waveform.squeeze(0).numpy().astype(np.float32)

    # --- Trim silence using librosa ---
    trimmed, _ = librosa.effects.trim(audio_np, top_db=50)
    audio_np = trimmed.astype(np.float32)

    # --- Apply light denoising ---
    reduced_noise = nr.reduce_noise(
        y=audio_np,
        sr=sample_rate,
        stationary=False,
        prop_decrease=0.7,
        n_fft=1024
    )

    # Normalize
    reduced_noise = reduced_noise / np.max(np.abs(reduced_noise) + 1e-6)

    # --- Save with torchaudio or soundfile ---
    sf.write(output_path, reduced_noise, sample_rate)
    print(f"Denoised audio saved to: {output_path}")

# Example usage:
denoise_audio(
    "/home/romolo/VT1/coqui-tts/data/TARGET.wav",
    "/home/romolo/VT1/coqui-tts/data/REF_no_noise.wav"
)
denoise_audio(
    "/home/romolo/VT1/coqui-tts/data/TARGET_5s.wav",
    "/home/romolo/VT1/coqui-tts/data/TARGET_5s_no_noise.wav"
)

Denoised audio saved to: /home/romolo/VT1/coqui-tts/data/REF_no_noise.wav
Denoised audio saved to: /home/romolo/VT1/coqui-tts/data/TARGET_5s_no_noise.wav


In [271]:
from TTS.tts.models.xtts import load_audio
from TTS.tts.layers.xtts.trainer.dataset import get_prompt_slice


lang = 'en'


audiopath = "/home/romolo/VT1/coqui-tts/data/TARGET_5s_no_noise.wav"
text = asr.transcribe(audiopath)['text']
tokens = train_model.xtts.tokenizer.encode(text, lang)
tseq = torch.IntTensor(tokens)
wav = load_audio(audiopath, 22050)
ref_sample = "/home/romolo/VT1/coqui-tts/data/REF_no_noise.wav"
cond, cond_len, _ = get_prompt_slice(
    ref_sample, model_args.max_conditioning_length, model_args.min_conditioning_length, 22050, True
)
cond_idxs = torch.nan

sample = {
    # 'real_text': text,
    "text": tseq,
    "text_lengths": torch.tensor(tseq.shape[0], dtype=torch.long),
    "wav": wav,
    "wav_lengths": torch.tensor(wav.shape[-1], dtype=torch.long),
    "filenames": audiopath,
    "conditioning": cond.unsqueeze(1),
    "cond_lens": torch.tensor(cond_len, dtype=torch.long)
    if cond_len is not torch.nan
    else torch.tensor([cond_len]),
    "cond_idxs": torch.tensor(cond_idxs) if cond_idxs is not torch.nan else torch.tensor([cond_idxs]),
}
sample

{'text': tensor([ 259,   95,    2,   32,   98,    7,    2,   22,    2,  235,    2,   27,
          474,    2,   80,   43,    2,   60,    2,  123,    7,    2,   53,    2,
           22,    2,  122,    2,   76, 2359,    7,    2,   76, 2359,    2, 1619,
         2359,    9], dtype=torch.int32),
 'text_lengths': tensor(38),
 'wav': tensor([[-0.0043, -0.0055, -0.0052,  ..., -0.0470,  0.0247,  0.0620]]),
 'wav_lengths': tensor(110250),
 'filenames': '/home/romolo/VT1/coqui-tts/data/TARGET_5s_no_noise.wav',
 'conditioning': tensor([[[0.0018, 0.0012, 0.0010,  ..., 0.0000, 0.0000, 0.0000]]]),
 'cond_lens': tensor(132300),
 'cond_idxs': tensor([nan])}

In [272]:
def collate_fn(batch):
    # convert list of dicts to dict of lists
    B = len(batch)

    batch = {k: [dic[k] for dic in batch] for k in batch[0]}

    # stack for features that already have the same shape
    batch["wav_lengths"] = torch.stack(batch["wav_lengths"])
    batch["text_lengths"] = torch.stack(batch["text_lengths"])
    batch["conditioning"] = torch.stack(batch["conditioning"])
    batch["cond_lens"] = torch.stack(batch["cond_lens"])
    batch["cond_idxs"] = torch.stack(batch["cond_idxs"])

    if torch.any(batch["cond_idxs"].isnan()):
        batch["cond_idxs"] = None

    if torch.any(batch["cond_lens"].isnan()):
        batch["cond_lens"] = None

    max_text_len = batch["text_lengths"].max()
    max_wav_len = batch["wav_lengths"].max()

    # create padding tensors
    text_padded = torch.IntTensor(B, max_text_len)
    wav_padded = torch.FloatTensor(B, 1, max_wav_len)

    # initialize tensors for zero padding
    text_padded = text_padded.zero_()
    wav_padded = wav_padded.zero_()
    for i in range(B):
        text = batch["text"][i]
        text_padded[i, : batch["text_lengths"][i]] = torch.IntTensor(text)
        wav = batch["wav"][i]
        wav_padded[i, :, : batch["wav_lengths"][i]] = torch.FloatTensor(wav)

    batch["wav"] = wav_padded
    batch["padded_text"] = text_padded
    return batch


collated = collate_fn([sample])
collated

{'text': [tensor([ 259,   95,    2,   32,   98,    7,    2,   22,    2,  235,    2,   27,
           474,    2,   80,   43,    2,   60,    2,  123,    7,    2,   53,    2,
            22,    2,  122,    2,   76, 2359,    7,    2,   76, 2359,    2, 1619,
          2359,    9], dtype=torch.int32)],
 'text_lengths': tensor([38]),
 'wav': tensor([[[-0.0043, -0.0055, -0.0052,  ..., -0.0470,  0.0247,  0.0620]]]),
 'wav_lengths': tensor([110250]),
 'filenames': ['/home/romolo/VT1/coqui-tts/data/TARGET_5s_no_noise.wav'],
 'conditioning': tensor([[[[0.0018, 0.0012, 0.0010,  ..., 0.0000, 0.0000, 0.0000]]]]),
 'cond_lens': tensor([132300]),
 'cond_idxs': None,
 'padded_text': tensor([[ 259,   95,    2,   32,   98,    7,    2,   22,    2,  235,    2,   27,
           474,    2,   80,   43,    2,   60,    2,  123,    7,    2,   53,    2,
            22,    2,  122,    2,   76, 2359,    7,    2,   76, 2359,    2, 1619,
          2359,    9]], dtype=torch.int32)}

In [273]:
train_model = train_model.to('cpu')
o = train_model.format_batch_on_device(collated)

In [274]:
cond_mels = o["cond_mels"].to(device)
text_inputs = o["text_inputs"].to(device)
text_lengths = o["text_lengths"].to(device)
audio_codes = o["audio_codes"].to(device)
wav_lengths = o["wav_lengths"].to(device)
cond_lens = o["cond_lens"].to(device)

train_model = train_model.to(device)

In [275]:
train_model.training = False
with torch.no_grad():
    (gpt_cond_latent, speaker_embedding) = model.get_conditioning_latents(
        audio_path=ref_sample,
        gpt_cond_len=45,
        gpt_cond_chunk_len=15,
        max_ref_length=45,
        sound_norm_refs=False,
    )


    gpt_latents = train_model.xtts.gpt(
        text_inputs,
        text_lengths,
        audio_codes,
        wav_lengths,
        cond_latents=gpt_cond_latent,
        return_attentions=False,
        return_latent=True,
    )

    gpt_latents.shape, speaker_embedding.shape

In [276]:
with torch.no_grad():
    owav = model.hifigan_decoder(gpt_latents, g=speaker_embedding).cpu().squeeze()

In [277]:
wav = model.forward_from_audios_and_text('en',text,audiopath,ref_sample,train_model,model_args.max_conditioning_length, model_args.min_conditioning_length)

In [278]:
import IPython.display as ipd


In [279]:
ipd.Audio(wav['wav'], rate=24000)

In [280]:
ipd.Audio(owav, rate=24000)

In [269]:
tts.synthesizer.save_wav(wav=wav['wav'], path="/home/romolo/VT1/coqui-tts/data/outputs/test2.wav")

In [139]:
tts.synthesizer.save_wav(wav=owav.numpy(), path="/home/romolo/VT1/coqui-tts/data/outputs/test.wav")

In [126]:
from huggingface_hub import hf_hub_download

# automatically checks for cached file, optionally set `cache_dir` location
model_file = hf_hub_download(repo_id='Jenthe/ECAPA2', filename='ecapa2.pt', cache_dir=None)

In [22]:
import torch
import torchaudio
import torch.nn.functional as F

ecapa2 = torch.jit.load(model_file, map_location='cuda')


In [23]:
#cossim between embeddings
audio, sr = torchaudio.load("/home/romolo/VT1/coqui-tts/data/outputs/test.wav")
audio = torchaudio.functional.resample(audio, orig_freq=sr, new_freq=16_000)# sample rate of 16 kHz expected
embedding = ecapa2(audio.to('cuda'))
ref_audio, sr = torchaudio.load(ref_sample) # sample rate of 16 kHz expected
ref_audio = torchaudio.functional.resample(ref_audio, orig_freq=sr, new_freq=16_000)
ref_embedding = ecapa2(ref_audio.to('cuda'))
sim = F.cosine_similarity(embedding, ref_embedding)
sim

tensor([0.4925], device='cuda:0')

In [24]:
#cossim between embeddings
audio, sr = torchaudio.load("/home/romolo/VT1/coqui-tts/data/outputs/test2.wav")
audio = torchaudio.functional.resample(audio, orig_freq=sr, new_freq=16_000)# sample rate of 16 kHz expected
embedding2 = ecapa2(audio.to('cuda'))
ref_audio, sr = torchaudio.load(ref_sample) # sample rate of 16 kHz expected
ref_audio = torchaudio.functional.resample(ref_audio, orig_freq=sr, new_freq=16_000)
ref_embedding = ecapa2(ref_audio.to('cuda'))
sim = F.cosine_similarity(embedding2, ref_embedding)
sim

tensor([0.5299], device='cuda:0')

In [25]:
target_audio, sr = torchaudio.load(audiopath) # sample rate of 16 kHz expected
target_audio = torchaudio.functional.resample(target_audio, orig_freq=sr, new_freq=16_000)
target_embedding = ecapa2(target_audio.to('cuda'))
sim = F.cosine_similarity(embedding2, target_embedding)
sim

tensor([0.1507], device='cuda:0')

In [26]:
target_audio, sr = torchaudio.load(audiopath) # sample rate of 16 kHz expected
target_audio = torchaudio.functional.resample(target_audio, orig_freq=sr, new_freq=16_000)
target_embedding = ecapa2(target_audio.to('cuda'))
sim = F.cosine_similarity(embedding, target_embedding)
sim

tensor([0.1367], device='cuda:0')

In [27]:
sim = F.cosine_similarity(ref_embedding, target_embedding)
sim

tensor([0.0990], device='cuda:0')